In [53]:
import pandas as pd
import numpy as np
import httpx
import json
import asyncio
from tqdm import tqdm
from agents import Agent, Runner, function_tool, set_tracing_disabled, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import os
import re

In [54]:
df = pd.read_excel("BREMEN.xlsx")

In [55]:
# Remove newline characters from column names
df.columns = df.columns.str.replace('\n', ' ')
df.columns

Index(['Unnamed: 0', 'Company name Latin alphabet', 'Country ISO code',
       'City Latin Alphabet', 'NACE Rev. 2, core code (4 digits)',
       'BvD ID number', 'NACE Rev. 2 main section', 'Region in country',
       'Status', 'Date of incorporation', 'Number of employees 2024',
       'Number of employees 2023', 'Number of employees 2022',
       'Number of employees 2021', 'Number of employees 2020',
       'Number of employees 2019', 'Number of employees 2018',
       'Number of employees 2017', 'Founded Year', 'Region in country clean',
       'Growth 2023', 'Growth 2022', 'Growth 2021', 'Growth 2020',
       'Growth 2019', 'Growth 2018', 'Growth 2024', 'aagr 2023', 'aagr 2022',
       'aagr 2021', 'aagr 2020', 'aagr 2024', 'Scaler 2023', 'Scaler 2022',
       'Scaler 2021', 'Scaler 2020', 'Scaler 2024', 'HighGrowthFirm 2023',
       'HighGrowthFirm 2022', 'HighGrowthFirm 2021', 'HighGrowthFirm 2020',
       'HighGrowthFirm 2024', 'ConsistentHighGrowthFirm 2023',
       'Consiste

In [56]:
# Clean n.a. and redefine data type
df.replace('n.a.', np.nan, inplace=True)
year_cols = [col for col in df.columns if any(yr in col for yr in ['2017','2018','2019','2020','2021','2022','2023','2024'])]
df[year_cols] = df[year_cols].apply(pd.to_numeric, errors='coerce')

/var/folders/7g/hwg5gctn1gl20zn6p9xc946h0000gp/T/ipykernel_27257/989795843.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('n.a.', np.nan, inplace=True)


In [57]:
# Remove status as "Dissolve", "In liquidation", "Status unknown"
df["Status"] = df["Status"].str.strip()
unwanted_status = ["Dissolve", "In liquidation", "Status unknown"]
df = df[~df["Status"].isin(unwanted_status)]

In [58]:
# For duplicate company names, keep the first one and drop the rest
df = df.drop_duplicates(subset='Company name Latin alphabet', keep='first')

In [59]:
# Get API keys
with open("groq.txt", "r", encoding="utf-8") as file:
    groq_api_key = file.readline().strip()

with open("serper.txt", "r", encoding="utf-8") as file:
    serper_api_key = file.readline().strip()

In [60]:
# Disable tracing since we're not using OpenAI API (to avoid 401 errors)
set_tracing_disabled(True)

In [61]:
# Define the Serper search tool using the @function_tool decorator
# This creates a tool that the agent can use to search the internet
# Returns condensed results to minimize token usage

@function_tool
async def search_internet(query: str) -> str:
    """Search the internet for information using Google Serper API.
    
    Args:
        query: The search query to look up on the internet.
    """
    url = "https://google.serper.dev/search"
    
    headers = {
        "X-API-KEY": serper_api_key,
        "Content-Type": "application/json"
    }
    
    payload = {
        "q": query,
        "num": 5  # Reduced from 10 to minimize tokens
    }
    
    try:
        async with httpx.AsyncClient() as client:
            response = await client.post(url, headers=headers, json=payload, timeout=30.0)
            response.raise_for_status()
            data = response.json()

        
        return data
    
    except Exception as e:
        return f"Error searching: {str(e)}"

In [62]:
# Create the agent using Groq via OpenAIChatCompletionsModel
# Using Llama 4 Scout - smaller and faster with better rate limits

groq_client = AsyncOpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)

company_research_agent = Agent(
    name="Company Research Agent",
    instructions="""
You are a company analyst performing factual research.

TASK:
Research the company explicitly provided by the user using the `search_internet` tool only.
Do NOT rely on prior knowledge or assumptions.

IMPORTANT SEARCH STRATEGY:
1. You MUST use the `search_internet` tool.
2. Minimize the number of tool calls.
3. Use at most 2 search queries total for one company.
4. Each search query should try to gather multiple fields at once.
5. Do NOT repeat similar searches unless clearly necessary.
6. If some fields remain unclear after limited searches, return "Not found" for those fields instead of continuing to search.

SOURCE RULES:
1. Rely only on information explicitly stated in reliable sources such as:
   - official company websites
   - official filings and registry records
   - reputable business databases
   - major news outlets
2. If a data point cannot be confidently verified, return "Not found" for that field.
3. Do NOT infer, estimate, assume, or fabricate any information.

DATA TO COLLECT:
- Legal_Form: Legal form explicitly stated in sources (e.g. GmbH, UG, AG, Ltd, Inc., LLC). Return exactly as stated.
- Company_Description: 1 to 2 sentence factual description based on reliable sources.
- Key_Activities_Product_Offerings: Core products or services explicitly mentioned by the company or other reliable sources.
- Industry: Industry or sector explicitly stated in sources. Do NOT map or normalize it yourself.
- B2B_or_B2C: Return B2B, B2C, Both, or Not found only if clearly supported by explicit source evidence.
  - Return "B2B" only if the company explicitly serves businesses, enterprises, professionals, or institutional clients.
  - Return "B2C" only if the company explicitly serves consumers or individual end users.
  - Return "Both" only if both are explicitly supported.
  - Otherwise return "Not found".

OUTPUT FORMAT:
Return only a JSON object with exactly these keys:
{
  "Legal_Form": "",
  "Company_Description": "",
  "Key_Activities_Product_Offerings": "",
  "Industry": "",
  "B2B_or_B2C": ""
}

STRICT CONSTRAINTS:
- Use "Not found" if information is unavailable, ambiguous, inferred, or unclear.
- Return only the JSON object.
- Do NOT add explanations, commentary, citations, markdown, or extra fields.
- Do NOT guess or generalize.
- Keep all outputs concise and factual.
""",
    model=OpenAIChatCompletionsModel(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        openai_client=groq_client,
    ),
    tools=[search_internet],
)

In [63]:
# Helper function to run the agent asynchronously with retry logic for rate limits

async def research_company(company_input: str, max_retries: int = 3) -> str:
    prompt = f"""Can you please help me to collect information 
for the following company: {company_input}"""
    
    for attempt in range(max_retries):
        try:
            output = await Runner.run(company_research_agent, prompt)
            return output
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # 1s, 2s, 4s
                print(f"⚠️ Retry {attempt+1}/{max_retries} for {company_input}, waiting {wait}s: {e}")
                await asyncio.sleep(wait)
            else:
                raise

In [64]:
# What data to collect from Internet?
# Funding round
# Money Raised (Total funding raised)
# Legal Form 
# B2B or B2C
# Founder's background
# Industry (To be cleaned and mapped with NACE Rev. 2 main section)
# R&D spent indicator (Innovation)



In [65]:
# Create a new dataframe for searching information from internet
df_agent = df[[
    'Company name Latin alphabet',
    'Country ISO code',
    'City Latin Alphabet',
    'BvD ID number',
    'NACE Rev. 2 main section',
    'Date of incorporation'
]].copy()

df_agent.rename(columns={
    'Company name Latin alphabet': 'company_name',
    'Country ISO code': 'country',
    'City Latin Alphabet': 'city',
    'BvD ID number': 'bvd_id',
    'NACE Rev. 2 main section': 'nace_section',
    'Date of incorporation': 'incorporation_date'
}, inplace=True)

df_agent['agent_input'] = (
    df_agent['company_name'] + 
    " (" + df_agent['country'] + ", " + df_agent['city'] + ")"
)

In [66]:
# Define result columns
result_cols = [
    "Legal_Form",
    "Company_Description",
    "Key_Activities_Product_Offerings",
    "Industry",
    "B2B_or_B2C",
]

for col in result_cols:
    df_agent[col] = pd.Series(dtype="object")

In [67]:
# Read result 
def parse_agent_output(output_text: str) -> dict:
    """
    Parse JSON string returned by the agent.
    If parsing fails, fill all fields with 'Not found'.
    """
    empty_result = {col: "Not found" for col in result_cols}
    
    try:
        text = re.sub(r"```(?:json)?|```", "", output_text).strip()
        data = json.loads(text)
        if not isinstance(data, dict):
            return empty_result
        
        parsed = {}
        for col in result_cols:
            parsed[col] = data.get(col, "Not found")
        return parsed
    
    except Exception as e:
        print(f"Parse error: {e}")
        return empty_result

In [69]:
# Agent usage to gather information. (First 15 companies)
output_file = "df_agent_progress.xlsx"

if os.path.exists(output_file):
    df_saved = pd.read_excel(output_file)
    df_agent.update(df_saved)
    print(f"📂 Loaded existing progress from {output_file}")

for i, idx in enumerate(tqdm(df_agent.index[:15], desc="Running first 15 companies")):

    if pd.notna(df_agent.loc[idx, "Legal_Form"]) and df_agent.loc[idx, "Legal_Form"] != "Not found":
        print(f"⏭️ Skipping already processed: {df_agent.loc[idx, 'company_name']}")
        continue
    
    company_input = (
        str(df_agent.loc[idx, 'company_name']) +
        '; HQ city: ' + str(df_agent.loc[idx, 'city']) +
        '; Country: ' + str(df_agent.loc[idx, 'country'])
    )
    
    try:
        result_text = await research_company(company_input)
        parsed_result = parse_agent_output(result_text)
        
        for col in result_cols:
            df_agent.loc[idx, col] = parsed_result[col]
        
        print(f"✅ Done: {df_agent.loc[idx, 'company_name']}")
    
    except Exception as e:
        print(f"❌ Error at row {idx}, company {df_agent.loc[idx, 'company_name']}: {e}")
        
        for col in result_cols:
            df_agent.loc[idx, col] = "Not found"
    
    if i % 25 == 0 and i != 0:
        df_agent.to_excel(output_file, index=False)
        print(f"💾 Saved progress at iteration {i}")

df_agent.to_excel(output_file, index=False)
print("✅ Final save completed")


📂 Loaded existing progress from df_agent_progress.xlsx


Running first 15 companies: 100%|██████████| 15/15 [00:00<00:00, 49306.08it/s]

⏭️ Skipping already processed: ARCELORMITTAL BREMEN GMBH
⏭️ Skipping already processed: BREMER LAGERHAUS-GESELLSCHAFT - AKTIENGESELLSCHAFT VON 1877
⏭️ Skipping already processed: ROEHLIG LOGISTICS GMBH & CO. KG
⏭️ Skipping already processed: GEWOBA AKTIENGESELLSCHAFT WOHNEN UND BAUEN
⏭️ Skipping already processed: SLOMAN NEPTUN SCHIFFAHRTS AG
⏭️ Skipping already processed: W. TIEMANN GMBH & CO. KG
⏭️ Skipping already processed: SAACKE GMBH
⏭️ Skipping already processed: NORDSEE NASSBAGGER-UND TIEFBAU-GESELLSCHAFT MIT BESCHRAENKTER HAFTUNG
⏭️ Skipping already processed: SWB AG
⏭️ Skipping already processed: WFB WIRTSCHAFTSFOERDERUNG BREMEN GMBH
⏭️ Skipping already processed: BREBAU GMBH
⏭️ Skipping already processed: GEWOSIE WOHNUNGSBAUGENOSSENSCHAFT BREMEN-NORD E.G.
⏭️ Skipping already processed: ESPABAU EISENBAHN SPAR- UND BAUVEREIN BREMEN EG
⏭️ Skipping already processed: MONACOR INTERNATIONAL GMBH & CO. KOMMANDITGESELLSCHAFT
⏭️ Skipping already processed: BREPARK GMBH
✅ Final save c

In [70]:
df_agent.head(15)


,company_name,country,city,bvd_id,nace_section,incorporation_date,agent_input,Legal_Form,Company_Description,Key_Activities_Product_Offerings,Industry,B2B_or_B2C
0,ARCELORMITTAL BREMEN GMBH,DE,BREMEN,DE2050318132,C - Manufacturing,24695,"ARCELORMITTAL BREMEN GMBH (DE, BREMEN)",GmbH,ArcelorMittal Bremen GmbH is a steelworks on t...,The company operates a steel plant with blast ...,Steel,B2B
1,BREMER LAGERHAUS-GESELLSCHAFT - AKTIENGESELLSC...,DE,BREMEN,DE2050004094,H - Transportation and storage,1877,BREMER LAGERHAUS-GESELLSCHAFT - AKTIENGESELLSC...,Aktiengesellschaft,BREMER LAGERHAUS-GESELLSCHAFT - AKTIENGESELLSC...,"Seaport-oriented logistics services, Automobil...",Logistics,B2B
2,ROEHLIG LOGISTICS GMBH & CO. KG,DE,BREMEN,DE2050031350,"M - Professional, scientific and technical act...",01/05/1852,"ROEHLIG LOGISTICS GMBH & CO. KG (DE, BREMEN)",GmbH & Co. KG,Röhlig Logistics is a logistics company specia...,"Sea freight, air freight, project logistics, a...",Logistics,B2B
3,GEWOBA AKTIENGESELLSCHAFT WOHNEN UND BAUEN,DE,BREMEN,DE2050400546,L - Real estate activities,12952,GEWOBA AKTIENGESELLSCHAFT WOHNEN UND BAUEN (DE...,Aktiengesellschaft,Gewoba Aktiengesellschaft Wohnen und Bauen pro...,"real estate services, commercial and residenti...",Not found,B2C
4,SLOMAN NEPTUN SCHIFFAHRTS AG,DE,BREMEN,DE2050001215,H - Transportation and storage,1873,"SLOMAN NEPTUN SCHIFFAHRTS AG (DE, BREMEN)",AG,SLOMAN NEPTUN Schiffahrts-AG engages in the pr...,shipping services of various goods,Water Transport/Shipping,B2B
5,W. TIEMANN GMBH & CO. KG,DE,BREMEN,DE2050002720,G - Wholesale and retail trade; repair of moto...,1905,"W. TIEMANN GMBH & CO. KG (DE, BREMEN)",GmbH & Co. KG,W. Tiemann GmbH & Co. KG deals with commercial...,"Commercial vehicles, purchase, advice, renting...",Sale of cars and light motor vehicles,B2B
6,SAACKE GMBH,DE,BREMEN,DE2050004232,"M - Professional, scientific and technical act...",18629,"SAACKE GMBH (DE, BREMEN)",GmbH,SAACKE GmbH is one of the world's leading supp...,The company develops and produces innovative s...,Other engineering activities and related techn...,B2B
7,NORDSEE NASSBAGGER-UND TIEFBAU-GESELLSCHAFT MI...,DE,BREMEN,DE2050726965,F - Construction,24888,NORDSEE NASSBAGGER-UND TIEFBAU-GESELLSCHAFT MI...,GmbH,"Execution of dredging, water construction and ...","Dredging, water construction and civil enginee...",Not found,B2B
8,SWB AG,DE,BREMEN,DE2050055194,K - Financial and insurance activities,15169,"SWB AG (DE, BREMEN)",AG,"swb AG provides utilities services in energy, ...","energy, water, public transport, waste management",utilities,B2B
9,WFB WIRTSCHAFTSFOERDERUNG BREMEN GMBH,DE,BREMEN,DE2050000763,O - Public administration and defence; compuls...,1924,"WFB WIRTSCHAFTSFOERDERUNG BREMEN GMBH (DE, BRE...",GmbH,WFB Wirtschaftsförderung Bremen GmbH (WFB Brem...,"funding programmes, innovation consulting, rea...",Not found,B2B


In [71]:
df_agent.loc[:14, ["company_name", "Legal_Form", "Company_Description", "Key_Activities_Product_Offerings", "Industry", "B2B_or_B2C"]]


,company_name,Legal_Form,Company_Description,Key_Activities_Product_Offerings,Industry,B2B_or_B2C
0,ARCELORMITTAL BREMEN GMBH,GmbH,ArcelorMittal Bremen GmbH is a steelworks on t...,The company operates a steel plant with blast ...,Steel,B2B
1,BREMER LAGERHAUS-GESELLSCHAFT - AKTIENGESELLSC...,Aktiengesellschaft,BREMER LAGERHAUS-GESELLSCHAFT - AKTIENGESELLSC...,"Seaport-oriented logistics services, Automobil...",Logistics,B2B
2,ROEHLIG LOGISTICS GMBH & CO. KG,GmbH & Co. KG,Röhlig Logistics is a logistics company specia...,"Sea freight, air freight, project logistics, a...",Logistics,B2B
3,GEWOBA AKTIENGESELLSCHAFT WOHNEN UND BAUEN,Aktiengesellschaft,Gewoba Aktiengesellschaft Wohnen und Bauen pro...,"real estate services, commercial and residenti...",Not found,B2C
4,SLOMAN NEPTUN SCHIFFAHRTS AG,AG,SLOMAN NEPTUN Schiffahrts-AG engages in the pr...,shipping services of various goods,Water Transport/Shipping,B2B
5,W. TIEMANN GMBH & CO. KG,GmbH & Co. KG,W. Tiemann GmbH & Co. KG deals with commercial...,"Commercial vehicles, purchase, advice, renting...",Sale of cars and light motor vehicles,B2B
6,SAACKE GMBH,GmbH,SAACKE GmbH is one of the world's leading supp...,The company develops and produces innovative s...,Other engineering activities and related techn...,B2B
7,NORDSEE NASSBAGGER-UND TIEFBAU-GESELLSCHAFT MI...,GmbH,"Execution of dredging, water construction and ...","Dredging, water construction and civil enginee...",Not found,B2B
8,SWB AG,AG,"swb AG provides utilities services in energy, ...","energy, water, public transport, waste management",utilities,B2B
9,WFB WIRTSCHAFTSFOERDERUNG BREMEN GMBH,GmbH,WFB Wirtschaftsförderung Bremen GmbH (WFB Brem...,"funding programmes, innovation consulting, rea...",Not found,B2B
